In [8]:
import geopandas as gpd
import os
import pandas as pd
from shapely.geometry import box

# === USER INPUTS ===
fields = gpd.read_file("path_to_shapefile_or_geojson_etc")
datecolumn = "your_date_column_name"  # Set to "" (empty string) if none
outputpath = "path_to_your_output_folder"  # Folder to store yearly shapefiles

# === CHECK OUTPUT FOLDER EXISTS ===
os.makedirs(outputpath, exist_ok=True)

# === FIX MISSING CRS ===
if fields.crs is None:
    fields.set_crs("EPSG:4326", inplace=True) 

# === ANONYMIZATION: BUFFER, DISSOLVE, EXPLODE ===
if fields.crs.is_geographic:
    raise ValueError("CRS is geographic (degrees). Reproject to a projected CRS in meters before buffering.")

fields_buffered = fields.copy()
fields_buffered["geometry"] = fields_buffered.geometry.buffer(3000)
fields_dissolved = fields_buffered.dissolve()
fields_exploded = fields_dissolved.explode(index_parts=False).reset_index(drop=True)

# === EXPORT BUFFERED POLYGONS ===
if datecolumn.strip() == "":
    output_file = os.path.join(outputpath, "buffered_fields_all.shp")
    fields_exploded.to_file(output_file)
    print(f"Exported entire buffered set: {output_file}")
else:
    fields_exploded["temp_id"] = range(len(fields_exploded))
    joined = gpd.sjoin(fields_exploded, fields, how="left", predicate="intersects")
    joined["year"] = pd.to_datetime(joined[datecolumn], errors="coerce").dt.year
    joined = joined.dropna(subset=["year"])
    joined["year"] = joined["year"].astype(int)

    for year, group in joined.groupby("year"):
        output_file = os.path.join(outputpath, f"buffered_fields_{year}.shp")
        group = group.drop(columns=["index_right", "temp_id"], errors="ignore")
        group.to_file(output_file)
        print(f"Exported: {output_file}")

# === FUNCTION: GRID OVER SINGLE POLYGON EXTENT ===
def grid_over_polygon(geom, grid_size):
    xmin, ymin, xmax, ymax = geom.bounds
    width = height = grid_size

    rows = int((ymax - ymin) // height) + 1
    cols = int((xmax - xmin) // width) + 1

    cells = []
    for i in range(cols):
        for j in range(rows):
            x0 = xmin + i * width
            y0 = ymin + j * height
            x1 = x0 + width
            y1 = y0 + height
            cell = box(x0, y0, x1, y1)
            cells.append(cell)
    return cells

# === MAIN LOGIC ===
if datecolumn.strip() == "":
    # No date column: dissolve, explode once
    fields_dissolved = fields_buffered.dissolve()
    fields_exploded = fields_dissolved.explode(index_parts=False).reset_index(drop=True)

    # Export buffered shapefile
    buffered_file = os.path.join(outputpath, "buffered_fields_all.shp")
    fields_exploded.to_file(buffered_file)
    print(f"Exported: {buffered_file}")

    # Generate grid for each polygon
    all_cells = []
    for idx, row in fields_exploded.iterrows():
        cells = grid_over_polygon(row.geometry, 2560)
        all_cells.extend(cells)

    grid_gdf = gpd.GeoDataFrame(geometry=all_cells, crs=fields.crs)
    grid_file = os.path.join(outputpath, "buffered_fields_grid_2560m.shp")
    grid_gdf.to_file(grid_file)
    print(f"Exported: {grid_file}")

else:
    # Parse year from date column
    fields["year"] = pd.to_datetime(fields[datecolumn], errors="coerce").dt.year
    fields = fields.dropna(subset=["year"])
    fields["year"] = fields["year"].astype(int)

    for year, group in fields.groupby("year"):
        print(f"Processing year: {year}")
        # Buffer, dissolve, explode for this year
        buffered = group.copy()
        buffered["geometry"] = buffered.geometry.buffer(3000)
        dissolved = buffered.dissolve()
        exploded = dissolved.explode(index_parts=False).reset_index(drop=True)

        # Export buffered polygons
        buffered_file = os.path.join(outputpath, f"buffered_fields_{year}.shp")
        exploded.to_file(buffered_file)
        print(f"Exported: {buffered_file}")

        # Create grid over each exploded polygon's extent
        all_cells = []
        for idx, row in exploded.iterrows():
            cells = grid_over_polygon(row.geometry, 2560)
            all_cells.extend(cells)

        grid_gdf = gpd.GeoDataFrame(geometry=all_cells, crs=fields.crs)
        grid_file = os.path.join(outputpath, f"buffered_fields_grid_2560m_{year}.shp")
        grid_gdf.to_file(grid_file)
        print(f"Exported: {grid_file}")

DataSourceError: path_to_shapefile_or_geojson_etc: No such file or directory